# Faz 7 · Qwen3-VL gerçek embedding üretimi

Bu notebook yalnız **gerçek 2048d** video/frame embedding üretir. Sentetik embedding veya kalite sayısı üretmez. Runtime → GPU seçip tüm hücreleri çalıştırın.

In [ ]:
import subprocess
import torch

try:
    print(subprocess.run(['nvidia-smi'], check=False, capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('nvidia-smi bulunamadı')
if not torch.cuda.is_available():
    raise SystemExit('GPU yok: Colab Runtime > Change runtime type > GPU seçip yeniden çalıştırın.')
GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
print({'gpu': GPU_NAME, 'capability': CAPABILITY})

In [ ]:
%pip install -q torch==2.4.1 torchvision==0.19.1 transformers==4.46.0 accelerate==0.34.2 qwen-vl-utils==0.0.8 huggingface-hub==1.24.0 pyarrow==17.0.0 pandas==2.2.3 numpy==1.26.4
!pip uninstall -y torchaudio -q
![ -d /content/Qwen3-VL-Embedding ] || git clone --depth 1 https://github.com/QwenLM/Qwen3-VL-Embedding.git /content/Qwen3-VL-Embedding
import sys
sys.path.insert(0, '/content/Qwen3-VL-Embedding')

In [ ]:
from huggingface_hub import snapshot_download
MODEL_ID = 'Qwen/Qwen3-VL-Embedding-2B'
MODEL_REVISION = 'main'  # Üretim tesliminde manifestte çözümlenen commit SHA saklanır.
MODEL_PATH = snapshot_download(MODEL_ID, revision=MODEL_REVISION, local_dir='/content/models/Qwen3-VL-Embedding-2B')
print(MODEL_PATH)

In [ ]:
import torch
from src.models.qwen3_vl_embedding import Qwen3VLEmbedder

major, _ = torch.cuda.get_device_capability(0)
if major < 8:  # T4/Turing
    TORCH_DTYPE = torch.float16
    ATTN_IMPL = 'sdpa'
else:  # Ampere+ (L4/A100/H100)
    TORCH_DTYPE = torch.bfloat16
    ATTN_IMPL = 'flash_attention_2'
model = Qwen3VLEmbedder(
    model_name_or_path=MODEL_PATH, fps=1.0, max_frames=8, max_length=16384,
    torch_dtype=TORCH_DTYPE, attn_implementation=ATTN_IMPL,
)
print({'dtype': str(TORCH_DTYPE), 'attention': ATTN_IMPL})

In [ ]:
from pathlib import Path
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
DATASET = 'auair'  # capera | auair | seadronessee
DRIVE_ROOT = Path('/content/drive/MyDrive/Multimodal-Video-Intelligence')
INPUT_MANIFEST = DRIVE_ROOT / 'inputs' / f'{DATASET}_embedding_inputs.parquet'
items = pd.read_parquet(INPUT_MANIFEST)
required = {'segment_id', 'media'}
if not required.issubset(items.columns):
    raise ValueError(f'Manifest kolonları eksik: {required - set(items.columns)}')
# media: video path string veya sekiz frame path içeren list. İkisi de Qwen process için geçerlidir.
print({'dataset': DATASET, 'items': len(items), 'manifest': str(INPUT_MANIFEST)})

In [ ]:
import json
import time
import numpy as np

FRAMES_PER_ITEM = 8
CHECKPOINT_EVERY = 200
OUT = DRIVE_ROOT / 'artifacts' / 'embeddings'
OUT.mkdir(parents=True, exist_ok=True)
partial_path = OUT / f'{DATASET}_2048.partial.npy'
done_path = OUT / f'{DATASET}_done_ids.json'
errors_path = OUT / f'{DATASET}_errors.jsonl'
done_ids = set(json.loads(done_path.read_text()) if done_path.exists() else [])
vectors = np.full((len(items), 2048), np.nan, dtype=np.float32)
if partial_path.exists():
    previous = np.load(partial_path)
    if previous.shape == vectors.shape:
        vectors[:] = previous
started = time.perf_counter()
completed_since_checkpoint = 0
for position, row in items.reset_index(drop=True).iterrows():
    segment_id = str(row.segment_id)
    if segment_id in done_ids:
        continue
    try:
        media = row.media.tolist() if hasattr(row.media, 'tolist') else row.media
        embedding = model.process([{'video': media}])[0].detach().cpu().float().numpy()
        embedding = embedding / np.linalg.norm(embedding)
        vectors[position] = embedding.astype(np.float32)
        done_ids.add(segment_id)
        completed_since_checkpoint += 1
    except Exception as exc:
        with errors_path.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps({'segment_id': segment_id, 'error': f'{type(exc).__name__}: {exc}'}, ensure_ascii=False) + '\n')
    if completed_since_checkpoint and completed_since_checkpoint % CHECKPOINT_EVERY == 0:
        np.save(partial_path, vectors)
        done_path.write_text(json.dumps(sorted(done_ids), ensure_ascii=False), encoding='utf-8')
        print({'checkpoint': len(done_ids), 'elapsed_s': round(time.perf_counter() - started, 1)})
np.save(partial_path, vectors)
done_path.write_text(json.dumps(sorted(done_ids), ensure_ascii=False), encoding='utf-8')
ELAPSED_S = time.perf_counter() - started

In [ ]:
import numpy as np

if len(done_ids) != len(items):
    raise AssertionError(f'Eksik embedding: {len(done_ids)}/{len(items)}; errors.jsonl dosyasını inceleyin')
assert vectors.shape == (len(items), 2048)
assert vectors.dtype == np.float32
assert np.isfinite(vectors).all()
norms = np.linalg.norm(vectors, axis=1)
assert np.allclose(norms, 1.0, atol=1e-5)
print({'shape': vectors.shape, 'dtype': str(vectors.dtype), 'nan_inf': 0, 'norm_max_error': float(np.max(np.abs(norms - 1)))})

In [ ]:
import json
import subprocess
import pandas as pd

final_vectors = OUT / f'{DATASET}_2048.npy'
final_ids = OUT / f'{DATASET}_ids.parquet'
np.save(final_vectors, vectors)
items[['segment_id']].to_parquet(final_ids, index=False)
try:
    revision = subprocess.check_output(['git', '-C', '/content/Qwen3-VL-Embedding', 'rev-parse', 'HEAD'], text=True).strip()
except Exception:
    revision = MODEL_REVISION
manifest = {
    'dataset_id': DATASET, 'count': len(items), 'dimension': 2048,
    'model_id': MODEL_ID, 'model_revision': revision, 'dtype': str(TORCH_DTYPE),
    'attention_implementation': ATTN_IMPL, 'gpu_name': torch.cuda.get_device_name(0),
    'frames_per_item': FRAMES_PER_ITEM, 'elapsed_s': ELAPSED_S, 'embedding_mode': 'real',
}
manifest_path = OUT / 'embedding_manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(manifest)

In [ ]:
import shutil
from google.colab import files

zip_base = Path('/content') / f'embeddings_{DATASET}'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=OUT))
drive_zip = DRIVE_ROOT / zip_path.name
shutil.copy2(zip_path, drive_zip)
print({'drive_zip': str(drive_zip), 'download_name': zip_path.name})
files.download(str(zip_path))